In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



In [14]:
# Load dataset
data = pd.read_csv("Dataset .csv")

data.head()


,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


The objective of this task is to develop a machine learning classification model that predicts the primary cuisine of a restaurant using available attributes such as location, pricing, ratings, and service features.

In [15]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [16]:
# Fill missing categorical values
data['Cuisines'] = data['Cuisines'].fillna('Unknown')
data['City'] = data['City'].fillna('Unknown')

# Fill missing numerical values
data['Average Cost for two'].fillna(data['Average Cost for two'].median())
data['Aggregate rating'].fillna(data['Aggregate rating'].median())
data['Votes'].fillna(0)


0        314
1        591
2        270
3        365
4        229
        ... 
9546     788
9547    1034
9548     661
9549     901
9550     591
Name: Votes, Length: 9551, dtype: int64

In [17]:
data['Primary_Cuisine'] = data['Cuisines'].apply(lambda x: x.split(',')[0])


In [18]:
selected_features = [
    'City',
    'Average Cost for two',
    'Price range',
    'Has Online delivery',
    'Has Table booking',
    'Aggregate rating',
    'Votes'
]

X = data[selected_features]
y = data['Primary_Cuisine']

A Random Forest classifier was selected for this task because it can handle non-linear relationships, works well with mixed feature types, and is robust to noise. It also performs better than linear models in multi-class classification problems with complex interactions.

In [19]:
categorical_features = ['City', 'Has Online delivery', 'Has Table booking']

numerical_features = [
    'Average Cost for two',
    'Price range',
    'Aggregate rating',
    'Votes'
]

preprocessor = ColumnTransformer(
    transformers = [
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
random_forest_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight='balanced'
    ))
])

random_forest_model.fit(X_train, y_train)

y_pred = random_forest_model.predict(X_test)

In [28]:
accuracy  = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

report = classification_report(y_test, y_pred, zero_division=0)
print("Classification Report: ",report)

Accuracy:  0.21925693354264783
Classification Report:                   precision    recall  f1-score   support

        Afghani       0.00      0.00      0.00         0
       American       0.15      0.22      0.18        46
         Andhra       0.00      0.00      0.00         1
        Arabian       0.00      0.00      0.00         0
          Asian       0.08      0.08      0.08        13
       Assamese       0.00      0.00      0.00         0
         Awadhi       0.00      0.00      0.00         0
            BBQ       0.00      0.00      0.00         3
         Bakery       0.20      0.21      0.20       112
       Bar Food       0.00      0.00      0.00         5
        Bengali       0.00      0.00      0.00         3
      Beverages       0.12      0.26      0.16        19
         Bihari       0.00      0.00      0.00         1
        Biryani       0.00      0.00      0.00        18
      Brazilian       0.50      0.67      0.57         3
      Breakfast       0.00      

The model was evaluated using accuracy, precision, recall, and F1-score. Accuracy provides an overall measure of correctness, while precision and recall offer class-wise insights. Due to the presence of multiple cuisine classes and class imbalance, F1-score is considered a more reliable metric.

In [29]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0, 10,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  1,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], dtype=int64)

The model performs well for popular cuisines such as North Indian, Cafe, and Bakery, which have a higher number of samples. However, cuisines with very few samples exhibit low recall and precision. This indicates that the model struggles to learn meaningful patterns for under-represented cuisines.

**Challenges and Biases Identified**

The dataset is highly imbalanced, with some cuisines having hundreds of samples while others have only one or two.

Rare cuisines suffer from poor recall due to insufficient training data.

Restaurants often serve multiple cuisines, but the model predicts only a single primary cuisine.

Structured numerical features may not fully capture culinary characteristics.